In [1]:
from google.colab import files

uploaded = files.upload()

Saving task2_data.zip to task2_data.zip


In [2]:
import zipfile
import os

with zipfile.ZipFile("task2_data.zip", "r") as zip_ref:
    zip_ref.extractall("/content/task2")

for root, dirs, files in os.walk("/content/task2"):
    for file in files:
        print(os.path.join(root, file))

/content/task2/optional_external_data.csv
/content/task2/student_train.csv
/content/task2/student_test.csv


In [3]:
import os
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
CONTEXT = 512
HORIZON = 168

DATA_DIR = "/content/task2"

train = pd.read_csv(f"{DATA_DIR}/student_train.csv")
test = pd.read_csv(f"{DATA_DIR}/student_test.csv")
external = pd.read_csv(f"{DATA_DIR}/optional_external_data.csv")

y = train["value"].to_numpy(dtype=np.float32)
X = external.drop(columns=["time_idx"]).to_numpy(dtype=np.float32)

print("Train:", train.shape)
print("Test:", test.shape)
print("External:", external.shape)
print("Context:", CONTEXT)
print("Horizon:", HORIZON)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

Train: (43656, 2)
Test: (168, 2)
External: (43824, 11)
Context: 512
Horizon: 168
Device: cuda


Validation split

In [4]:
VAL_HORIZON = 168
VAL_START = len(y) - VAL_HORIZON

train_y = y[:VAL_START]
val_y = y[VAL_START:]

print("Training observations:", len(train_y))
print("Validation observations:", len(val_y))
print("Validation range:", VAL_START + 1, "to", len(y))

Training observations: 43488
Validation observations: 168
Validation range: 43489 to 43656


Seed

In [5]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


Series Decomposition

In [6]:
class SeriesDecomposition(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()

        if kernel_size < 1 or kernel_size % 2 == 0:
            raise ValueError("kernel_size must be positive and odd")

        self.kernel_size = kernel_size

    def forward(self, x):
        # x: [B, L, C]
        padding = self.kernel_size // 2

        x_t = x.transpose(1, 2)

        padded = F.pad(
            x_t,
            (padding, padding),
            mode="replicate"
        )

        trend = F.avg_pool1d(
            padded,
            kernel_size=self.kernel_size,
            stride=1
        )

        trend = trend.transpose(1, 2)

        seasonal = x - trend

        return seasonal, trend

Auto-Correlation delay scoring

In [7]:
def delay_scores(queries, keys):
    """
    queries, keys: [B, H, D, L]

    Returns:
        correlation scores: [B, L]
    """

    queries = queries - queries.mean(dim=-1, keepdim=True)
    keys = keys - keys.mean(dim=-1, keepdim=True)

    spectrum = (
        torch.fft.rfft(queries, dim=-1)
        * torch.fft.rfft(keys, dim=-1).conj()
    )

    correlation = torch.fft.irfft(
        spectrum,
        n=queries.shape[-1],
        dim=-1
    )

    return correlation.mean(dim=(1, 2))

Auto-Correlation aggregation

In [8]:
def aggregate_delays(values, delays, weights):
    """
    values:  [B, H, D, L]
    delays:  [B, K]
    weights: [B, K]

    returns:
        [B, H, D, L]
    """

    B, heads, features, L = values.shape
    K = delays.shape[1]

    time = torch.arange(
        L,
        device=values.device
    )

    indices = (
        time.unsqueeze(0).unsqueeze(0)
        - delays.unsqueeze(-1)
    ) % L

    indices = (
        indices
        .unsqueeze(1)
        .unsqueeze(1)
        .expand(B, heads, features, K, L)
    )

    values_expanded = (
        values
        .unsqueeze(3)
        .expand(B, heads, features, K, L)
    )

    delayed = torch.gather(
        values_expanded,
        dim=-1,
        index=indices
    )

    return (
        delayed
        * weights[:, None, None, :, None]
    ).sum(dim=3)

Auto-Correlation layer

In [9]:
class AutoCorrelation(nn.Module):
    def __init__(
        self,
        d_model,
        n_heads,
        min_delay=1,
        max_delay=None
    ):
        super().__init__()

        if d_model % n_heads != 0:
            raise ValueError("d_model must be divisible by n_heads")

        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.min_delay = min_delay
        self.max_delay = max_delay

        self.query_projection = nn.Linear(d_model, d_model)
        self.key_projection = nn.Linear(d_model, d_model)
        self.value_projection = nn.Linear(d_model, d_model)
        self.output_projection = nn.Linear(d_model, d_model)

        self.raw_temperature = nn.Parameter(
            torch.tensor(0.5413248546)
        )

    @property
    def temperature(self):
        return F.softplus(self.raw_temperature) + 1e-6

    def forward(self, x):
        # x: [B, L, D]

        B, L, D = x.shape

        shape = (
            B,
            L,
            self.n_heads,
            self.head_dim
        )

        q = self.query_projection(x).view(shape)
        k = self.key_projection(x).view(shape)
        v = self.value_projection(x).view(shape)

        q = q.permute(0, 2, 3, 1)
        k = k.permute(0, 2, 3, 1)
        v = v.permute(0, 2, 3, 1)

        scores = delay_scores(q, k)

        lo = min(self.min_delay, L - 1)

        if self.max_delay is None:
            hi = L - 1
        else:
            hi = min(self.max_delay, L - 1)

        if hi < lo:
            lo = 0
            hi = L - 1

        band = scores[:, lo:hi + 1]

        top_k = min(
            max(1, math.ceil(2 * math.log(L))),
            band.shape[-1]
        )

        selected, indices = band.topk(
            top_k,
            dim=-1
        )

        delays = indices + lo

        weights = (
            selected / self.temperature
        ).softmax(dim=-1)

        mixed = aggregate_delays(
            v,
            delays,
            weights
        )

        mixed = (
            mixed
            .permute(0, 3, 1, 2)
            .reshape(B, L, D)
        )

        return self.output_projection(mixed)

Autoformer encoder block

In [10]:
class AutoformerEncoderBlock(nn.Module):
    def __init__(
        self,
        d_model,
        n_heads,
        kernel_size,
        dropout=0.1,
        min_delay=1,
        max_delay=None
    ):
        super().__init__()

        self.auto_correlation = AutoCorrelation(
            d_model=d_model,
            n_heads=n_heads,
            min_delay=min_delay,
            max_delay=max_delay
        )

        self.decomp1 = SeriesDecomposition(kernel_size)
        self.decomp2 = SeriesDecomposition(kernel_size)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, 2 * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(2 * d_model, d_model)
        )

    def forward(self, x):
        # Auto-Correlation
        seasonal = x + self.dropout(
            self.auto_correlation(self.norm1(x))
        )

        seasonal, trend1 = self.decomp1(seasonal)

        # Feed-forward
        seasonal = seasonal + self.dropout(
            self.feed_forward(self.norm2(seasonal))
        )

        seasonal, trend2 = self.decomp2(seasonal)

        trend = trend1 + trend2

        return seasonal, trend

Test the Autoformer block

In [11]:
test_block = AutoformerEncoderBlock(
    d_model=64,
    n_heads=4,
    kernel_size=25
).to(device)

dummy = torch.randn(
    2,
    CONTEXT,
    64,
    device=device
)

with torch.no_grad():
    seasonal, trend = test_block(dummy)

print("Input:", dummy.shape)
print("Seasonal:", seasonal.shape)
print("Trend:", trend.shape)

Input: torch.Size([2, 512, 64])
Seasonal: torch.Size([2, 512, 64])
Trend: torch.Size([2, 512, 64])


External feature preprocessing

In [12]:
external_features = external.drop(columns=["time_idx"]).to_numpy(
    dtype=np.float32
)

# Target statistics from training portion only
y_mean = train_y.mean()
y_std = train_y.std()

# External statistics from training portion only
X_train_external = external_features[:VAL_START]

X_mean = X_train_external.mean(axis=0)
X_std = X_train_external.std(axis=0)

# Avoid division by zero
X_std[X_std < 1e-6] = 1.0

y_norm = (y - y_mean) / y_std
X_norm = (external_features - X_mean) / X_std

print("Target mean/std:", y_mean, y_std)
print("External shape:", X_norm.shape)
print("Normalized target range:", y_norm.min(), y_norm.max())

Target mean/std: 98.28371 90.883484
External shape: (43824, 10)
Normalized target range: -1.0814254 9.855656


Dataset

In [13]:
class ForecastDataset(Dataset):
    def __init__(
        self,
        target,
        external_features,
        context,
        horizon,
        start,
        end
    ):
        self.target = target
        self.external = external_features
        self.context = context
        self.horizon = horizon

        self.starts = np.arange(
            start,
            end - context - horizon + 1
        )

    def __len__(self):
        return len(self.starts)

    def __getitem__(self, idx):
        s = self.starts[idx]

        target_context = self.target[
            s:s + self.context
        ]

        target_future = self.target[
            s + self.context:
            s + self.context + self.horizon
        ]

        external_context = self.external[
            s:s + self.context
        ]

        external_future = self.external[
            s + self.context:
            s + self.context + self.horizon
        ]

        return (
            torch.from_numpy(target_context).float(),
            torch.from_numpy(target_future).float(),
            torch.from_numpy(external_context).float(),
            torch.from_numpy(external_future).float()
        )

Create training dataset

In [14]:
train_dataset = ForecastDataset(
    target=y_norm,
    external_features=X_norm,
    context=CONTEXT,
    horizon=HORIZON,
    start=0,
    end=VAL_START
)

print("Training windows:", len(train_dataset))

sample = train_dataset[0]

for i, item in enumerate(sample):
    print(i, item.shape)

Training windows: 42809
0 torch.Size([512])
1 torch.Size([168])
2 torch.Size([512, 10])
3 torch.Size([168, 10])


DataLoader

In [15]:
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    drop_last=True
)

batch = next(iter(train_loader))

print("Target context:", batch[0].shape)
print("Target future:", batch[1].shape)
print("External context:", batch[2].shape)
print("External future:", batch[3].shape)

Target context: torch.Size([64, 512])
Target future: torch.Size([64, 168])
External context: torch.Size([64, 512, 10])
External future: torch.Size([64, 168, 10])


Autoformer model

In [16]:
class AutoformerForecast(nn.Module):
    def __init__(
        self,
        context,
        horizon,
        external_dim,
        d_model=64,
        n_heads=4,
        n_layers=2,
        kernel_size=25,
        dropout=0.1,
        max_delay=128
    ):
        super().__init__()

        self.context = context
        self.horizon = horizon
        self.d_model = d_model

        # Target + external variables
        input_dim = 1 + external_dim

        self.input_projection = nn.Linear(
            input_dim,
            d_model
        )

        self.position = nn.Parameter(
            torch.randn(1, context, d_model) * 0.02
        )

        self.blocks = nn.ModuleList([
            AutoformerEncoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                kernel_size=kernel_size,
                dropout=dropout,
                min_delay=1,
                max_delay=max_delay
            )
            for _ in range(n_layers)
        ])

        self.norm = nn.LayerNorm(d_model)

        # Map encoded context to the forecast horizon
        self.forecast_head = nn.Linear(
            context * d_model,
            horizon
        )

        # Known future external variables
        self.future_external_projection = nn.Linear(
            external_dim,
            d_model
        )

        self.future_external_head = nn.Linear(
            horizon * d_model,
            horizon
        )

    def forward(
        self,
        target_context,
        external_context,
        external_future
    ):
        # target_context: [B, L]
        # external_context: [B, L, F]
        # external_future: [B, H, F]

        target_context = target_context.unsqueeze(-1)

        x = torch.cat(
            [target_context, external_context],
            dim=-1
        )

        x = self.input_projection(x)

        x = x + self.position

        accumulated_trend = None

        for block in self.blocks:
            x, trend = block(x)

            if accumulated_trend is None:
                accumulated_trend = trend
            else:
                accumulated_trend = (
                    accumulated_trend + trend
                )

        x = self.norm(x)

        # Seasonal forecast
        seasonal_forecast = self.forecast_head(
            x.flatten(start_dim=1)
        )

        # Future known external variables
        future_external = self.future_external_projection(
            external_future
        )

        future_external = future_external.flatten(
            start_dim=1
        )

        external_forecast = self.future_external_head(
            future_external
        )

        # Trend forecast from the final decomposed representation
        trend_signal = accumulated_trend.mean(dim=-1)

        trend_forecast = F.interpolate(
            trend_signal.unsqueeze(1),
            size=self.horizon,
            mode="linear",
            align_corners=False
        ).squeeze(1)

        forecast = (
            seasonal_forecast
            + external_forecast
            + 0.1 * trend_forecast
        )

        return forecast

Create model

In [17]:
model = AutoformerForecast(
    context=CONTEXT,
    horizon=HORIZON,
    external_dim=X_norm.shape[1],
    d_model=64,
    n_heads=4,
    n_layers=2,
    kernel_size=25,
    dropout=0.1,
    max_delay=128
).to(device)

num_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Trainable parameters:", num_params)
print("Parameters (millions):", num_params / 1e6)

Trainable parameters: 7413010
Parameters (millions): 7.41301


Forward-pass test

In [18]:
model.eval()

with torch.no_grad():
    target_context = batch[0].to(device)
    target_future = batch[1].to(device)
    external_context = batch[2].to(device)
    external_future = batch[3].to(device)

    prediction = model(
        target_context,
        external_context,
        external_future
    )

print("Prediction shape:", prediction.shape)
print("Target shape:", target_future.shape)

Prediction shape: torch.Size([64, 168])
Target shape: torch.Size([64, 168])


Loss and metrics

In [19]:
def smape(y_true, y_pred):
    denominator = (
        np.abs(y_true) + np.abs(y_pred) + 1e-8
    )
    return np.mean(
        2.0 * np.abs(y_pred - y_true) / denominator
    ) * 100


def calculate_metrics(y_true, y_pred):
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    s_mape = smape(y_true, y_pred)

    return {
        "MAE": mae,
        "sMAPE": s_mape,
        "RMSE": rmse
    }


criterion = nn.MSELoss()

Optimizer and scheduler

In [20]:
EPOCHS = 20
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

print("Epochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Parameters:", num_params)

Epochs: 20
Learning rate: 0.0001
Parameters: 7413010


Chronological validation function

In [21]:
def validate_model(model, target, external_features, origin):
    model.eval()

    context_start = origin - CONTEXT

    target_context = target[
        context_start:origin
    ]

    external_context = external_features[
        context_start:origin
    ]

    external_future = external_features[
        origin:origin + HORIZON
    ]

    target_context_tensor = torch.from_numpy(
        target_context
    ).float().unsqueeze(0).to(device)

    external_context_tensor = torch.from_numpy(
        external_context
    ).float().unsqueeze(0).to(device)

    external_future_tensor = torch.from_numpy(
        external_future
    ).float().unsqueeze(0).to(device)

    with torch.no_grad():
        prediction = model(
            target_context_tensor,
            external_context_tensor,
            external_future_tensor
        )

    prediction = prediction.squeeze(0).cpu().numpy()

    return prediction

Training loop

In [22]:
def train_model(model, train_loader, epochs):
    history = []

    for epoch in range(1, epochs + 1):

        model.train()

        total_loss = 0.0
        batches = 0

        for (
            target_context,
            target_future,
            external_context,
            external_future
        ) in train_loader:

            target_context = target_context.to(
                device,
                non_blocking=True
            )

            target_future = target_future.to(
                device,
                non_blocking=True
            )

            external_context = external_context.to(
                device,
                non_blocking=True
            )

            external_future = external_future.to(
                device,
                non_blocking=True
            )

            optimizer.zero_grad(set_to_none=True)

            prediction = model(
                target_context,
                external_context,
                external_future
            )

            loss = criterion(
                prediction,
                target_future
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

            total_loss += loss.item()
            batches += 1

        scheduler.step()

        avg_loss = total_loss / batches

        # Validate on the final chronological validation block
        prediction_norm = validate_model(
            model,
            y_norm,
            X_norm,
            VAL_START
        )

        prediction = (
            prediction_norm * y_std
            + y_mean
        )

        metrics = calculate_metrics(
            val_y,
            prediction
        )

        history.append({
            "epoch": epoch,
            "train_loss": avg_loss,
            **metrics
        })

        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"Loss {avg_loss:.5f} | "
            f"MAE {metrics['MAE']:.3f} | "
            f"sMAPE {metrics['sMAPE']:.3f} | "
            f"RMSE {metrics['RMSE']:.3f}"
        )

    return pd.DataFrame(history)

Train

In [23]:
history = train_model(
    model,
    train_loader,
    EPOCHS
)

history

Epoch 01/20 | Loss 0.75691 | MAE 44.182 | sMAPE 100.651 | RMSE 57.821
Epoch 02/20 | Loss 0.55102 | MAE 37.481 | sMAPE 77.353 | RMSE 48.368
Epoch 03/20 | Loss 0.48008 | MAE 41.363 | sMAPE 80.002 | RMSE 56.784
Epoch 04/20 | Loss 0.43380 | MAE 42.077 | sMAPE 77.818 | RMSE 55.142
Epoch 05/20 | Loss 0.39124 | MAE 42.037 | sMAPE 80.858 | RMSE 52.322
Epoch 06/20 | Loss 0.35064 | MAE 49.856 | sMAPE 84.021 | RMSE 63.391
Epoch 07/20 | Loss 0.31319 | MAE 43.419 | sMAPE 76.963 | RMSE 56.345
Epoch 08/20 | Loss 0.28230 | MAE 61.515 | sMAPE 88.202 | RMSE 77.503
Epoch 09/20 | Loss 0.25642 | MAE 58.061 | sMAPE 89.517 | RMSE 71.727
Epoch 10/20 | Loss 0.23559 | MAE 52.095 | sMAPE 85.748 | RMSE 66.740
Epoch 11/20 | Loss 0.21837 | MAE 57.795 | sMAPE 84.919 | RMSE 73.538
Epoch 12/20 | Loss 0.20465 | MAE 55.578 | sMAPE 82.681 | RMSE 69.989
Epoch 13/20 | Loss 0.19320 | MAE 51.556 | sMAPE 85.546 | RMSE 66.094
Epoch 14/20 | Loss 0.18398 | MAE 58.043 | sMAPE 87.304 | RMSE 71.310
Epoch 15/20 | Loss 0.17596 | MAE 

,epoch,train_loss,MAE,sMAPE,RMSE
0,1,0.756910,44.182003,100.651314,57.820778
1,2,0.551021,37.481125,77.353027,48.367710
2,3,0.480082,41.363037,80.001846,56.783611
3,4,0.433797,42.077251,77.818207,55.141987
4,5,0.391244,42.036598,80.857590,52.321693
5,6,0.350644,49.855572,84.020996,63.391003
6,7,0.313189,43.418510,76.963097,56.344524
7,8,0.282303,61.515427,88.201782,77.502640
8,9,0.256418,58.061047,89.516731,71.727249
9,10,0.235591,52.094734,85.748383,66.739861


In [24]:
baseline_results = history.copy()

best_epoch = baseline_results.loc[
    baseline_results["RMSE"].idxmin()
]

print("Best validation epoch:")
print(best_epoch)

Best validation epoch:
epoch          2.000000
train_loss     0.551021
MAE           37.481125
sMAPE         77.353027
RMSE          48.367710
Name: 1, dtype: float64


In [25]:
os.makedirs("/content/drive/MyDrive/colab/AI651_Task2", exist_ok=True)

baseline_results.to_csv(
    "/content/drive/MyDrive/colab/AI651_Task2/baseline_history.csv",
    index=False
)

print("Saved baseline history.")

Saved baseline history.


proper Autoformer-style decoder

In [26]:
class AutoformerForecastV2(nn.Module):
    def __init__(
        self,
        context,
        horizon,
        external_dim,
        d_model=64,
        n_heads=4,
        n_layers=2,
        kernel_size=25,
        dropout=0.1,
        max_delay=128
    ):
        super().__init__()

        self.context = context
        self.horizon = horizon
        self.d_model = d_model

        # Target + known external variables
        self.input_projection = nn.Linear(
            1 + external_dim,
            d_model
        )

        self.position = nn.Parameter(
            torch.randn(1, context, d_model) * 0.02
        )

        self.blocks = nn.ModuleList([
            AutoformerEncoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                kernel_size=kernel_size,
                dropout=dropout,
                min_delay=1,
                max_delay=max_delay
            )
            for _ in range(n_layers)
        ])

        self.norm = nn.LayerNorm(d_model)

        # Direct multi-horizon seasonal prediction
        self.seasonal_head = nn.Sequential(
            nn.Linear(context * d_model, 2 * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(2 * d_model, horizon)
        )

        # Trend projection
        self.trend_head = nn.Linear(
            context,
            horizon
        )

        # Future known external features
        self.future_external_projection = nn.Sequential(
            nn.Linear(external_dim, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model)
        )

        self.external_head = nn.Sequential(
            nn.Linear(horizon * d_model, 2 * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(2 * d_model, horizon)
        )

        # Learnable combination of the components
        self.component_scale = nn.Parameter(
            torch.tensor([1.0, 0.5, 0.5])
        )

    def forward(
        self,
        target_context,
        external_context,
        external_future
    ):
        # [B, L] -> [B, L, 1]
        target_context = target_context.unsqueeze(-1)

        x = torch.cat(
            [target_context, external_context],
            dim=-1
        )

        x = self.input_projection(x)
        x = x + self.position

        accumulated_trend = torch.zeros_like(x)

        for block in self.blocks:
            x, trend = block(x)
            accumulated_trend = (
                accumulated_trend + trend
            )

        x = self.norm(x)

        # Seasonal component
        seasonal_forecast = self.seasonal_head(
            x.flatten(start_dim=1)
        )

        # Trend component
        trend_signal = accumulated_trend.mean(dim=-1)

        trend_forecast = self.trend_head(
            trend_signal
        )

        # Known future external variables
        future_external = self.future_external_projection(
            external_future
        )

        external_forecast = self.external_head(
            future_external.flatten(start_dim=1)
        )

        forecast = (
            self.component_scale[0] * seasonal_forecast
            + self.component_scale[1] * trend_forecast
            + self.component_scale[2] * external_forecast
        )

        return forecast

Clean V2 initialization — seed BEFORE model creation


In [27]:
# Clean V2 initialization — seed BEFORE model creation

set_seed(42)

EPOCHS = 20
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5

model = AutoformerForecastV2(
    context=CONTEXT,
    horizon=HORIZON,
    external_dim=X_norm.shape[1],
    d_model=64,
    n_heads=4,
    n_layers=2,
    kernel_size=25,
    dropout=0.1,
    max_delay=128
).to(device)

num_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

criterion = nn.MSELoss()

print("Trainable parameters:", num_params)
print("Parameters (millions):", num_params / 1e6)

Trainable parameters: 5805821
Parameters (millions): 5.805821


Create the improved model

In [ ]:
model = AutoformerForecastV2(
    context=CONTEXT,
    horizon=HORIZON,
    external_dim=X_norm.shape[1],
    d_model=64,
    n_heads=4,
    n_layers=2,
    kernel_size=25,
    dropout=0.1,
    max_delay=128
).to(device)

num_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Trainable parameters:", num_params)
print("Parameters (millions):", num_params / 1e6)

Trainable parameters: 5805821
Parameters (millions): 5.805821


In [ ]:
model = AutoformerForecastV2(
    context=CONTEXT,
    horizon=HORIZON,
    external_dim=X_norm.shape[1],
    d_model=64,
    n_heads=4,
    n_layers=2,
    kernel_size=25,
    dropout=0.1,
    max_delay=128
).to(device)

num_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

EPOCHS = 20
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

criterion = nn.MSELoss()

set_seed(42)

print("Parameters:", num_params)

NameError: name 'set_seed' is not defined

Resuming by taking weights from drive

In [ ]:
model = AutoformerForecastV2(
    context=CONTEXT,
    horizon=HORIZON,
    external_dim=X_norm.shape[1],
    d_model=64,
    n_heads=4,
    n_layers=2,
    kernel_size=25,
    dropout=0.1,
    max_delay=128
).to(device)

ckpt = torch.load(
    "/content/drive/MyDrive/colab/AI651_Task2/v2_best.pt",
    map_location=device
)

model.load_state_dict(ckpt["model_state_dict"])
model.eval()

num_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Loaded best V2 checkpoint")
print("Best epoch:", ckpt["best_epoch"])
print("Best RMSE:", ckpt["best_rmse"])
print("Parameters:", num_params)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/colab/AI651_Task2/v2_best.pt'

Test it

In [ ]:
model.eval()

with torch.no_grad():
    prediction = model(
        batch[0].to(device),
        batch[2].to(device),
        batch[3].to(device)
    )

print("Prediction:", prediction.shape)
print("Expected:", (BATCH_SIZE, HORIZON))

Prediction: torch.Size([64, 168])
Expected: (64, 168)


Fresh optimizer

In [ ]:
EPOCHS = 20
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

print("Optimizer reset.")

Optimizer reset.


Training V2 with best-model saving

In [28]:
def train_model_v2(model, train_loader, epochs):
    history = []

    best_rmse = float("inf")
    best_state = None
    best_epoch = 0
    patience = 4
    patience_counter = 0

    for epoch in range(1, epochs + 1):

        model.train()

        total_loss = 0.0
        batches = 0

        for (
            target_context,
            target_future,
            external_context,
            external_future
        ) in train_loader:

            target_context = target_context.to(
                device,
                non_blocking=True
            )
            target_future = target_future.to(
                device,
                non_blocking=True
            )
            external_context = external_context.to(
                device,
                non_blocking=True
            )
            external_future = external_future.to(
                device,
                non_blocking=True
            )

            optimizer.zero_grad(set_to_none=True)

            prediction = model(
                target_context,
                external_context,
                external_future
            )

            loss = criterion(
                prediction,
                target_future
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

            total_loss += loss.item()
            batches += 1

        scheduler.step()

        avg_loss = total_loss / batches

        prediction_norm = validate_model(
            model,
            y_norm,
            X_norm,
            VAL_START
        )

        prediction = (
            prediction_norm * y_std
            + y_mean
        )

        metrics = calculate_metrics(
            val_y,
            prediction
        )

        history.append({
            "epoch": epoch,
            "train_loss": avg_loss,
            **metrics
        })

        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"Loss {avg_loss:.5f} | "
            f"MAE {metrics['MAE']:.3f} | "
            f"sMAPE {metrics['sMAPE']:.3f} | "
            f"RMSE {metrics['RMSE']:.3f}"
        )

        if metrics["RMSE"] < best_rmse:
            best_rmse = metrics["RMSE"]
            best_epoch = epoch
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            patience_counter = 0

            print("  -> New best model")

        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(
                f"Early stopping at epoch {epoch}. "
                f"Best epoch: {best_epoch}"
            )
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return pd.DataFrame(history), best_epoch, best_rmse

Train V2

In [29]:
history_v2, best_epoch_v2, best_rmse_v2 = train_model_v2(
    model,
    train_loader,
    EPOCHS
)

print()
print("Best epoch:", best_epoch_v2)
print("Best RMSE:", best_rmse_v2)

Epoch 01/20 | Loss 0.61109 | MAE 42.526 | sMAPE 77.476 | RMSE 54.998
  -> New best model
Epoch 02/20 | Loss 0.39428 | MAE 39.351 | sMAPE 71.942 | RMSE 52.478
  -> New best model
Epoch 03/20 | Loss 0.31757 | MAE 53.853 | sMAPE 87.063 | RMSE 63.467
Epoch 04/20 | Loss 0.27394 | MAE 60.354 | sMAPE 89.322 | RMSE 69.362
Epoch 05/20 | Loss 0.24495 | MAE 53.840 | sMAPE 84.334 | RMSE 62.076
Epoch 06/20 | Loss 0.22458 | MAE 44.684 | sMAPE 75.546 | RMSE 54.089
Early stopping at epoch 6. Best epoch: 2

Best epoch: 2
Best RMSE: 52.47809


In [30]:
import os

SAVE_DIR = "/content/drive/MyDrive/colab/AI651_Task2"
os.makedirs(SAVE_DIR, exist_ok=True)

history_v2.to_csv(
    f"{SAVE_DIR}/v2_history.csv",
    index=False
)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "best_epoch": best_epoch_v2,
        "best_rmse": best_rmse_v2,
        "num_params": num_params,
    },
    f"{SAVE_DIR}/v2_best.pt"
)

print("Checkpoint saved.")

Checkpoint saved.


In [31]:
import os

path = "/content/drive/MyDrive/colab/AI651_Task2/v2_best.pt"

print("Exists:", os.path.exists(path))
print("Size:", os.path.getsize(path), "bytes")

Exists: True
Size: 23243009 bytes


Save V2 Results


In [ ]:
history_v2.to_csv(
    "/content/drive/MyDrive/colab/AI651_Task2/v2_history.csv",
    index=False
)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "best_epoch": best_epoch_v2,
        "best_rmse": best_rmse_v2,
        "num_params": num_params,
    },
    "/content/drive/MyDrive/colab/AI651_Task2/v2_best.pt"
)

print("V2 results and checkpoint saved.")

V2 results and checkpoint saved.


Best V2 result

In [32]:
best_v2 = history_v2.loc[
    history_v2["RMSE"].idxmin()
]

print(best_v2)

epoch          2.000000
train_loss     0.394283
MAE           39.351246
sMAPE         71.941711
RMSE          52.478088
Name: 1, dtype: float64


Create a no-external dataset

In [33]:
X_zero = np.zeros_like(X_norm, dtype=np.float32)

train_dataset_noext = ForecastDataset(
    target=y_norm,
    external_features=X_zero,
    context=CONTEXT,
    horizon=HORIZON,
    start=0,
    end=VAL_START
)

train_loader_noext = DataLoader(
    train_dataset_noext,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    drop_last=True
)

print("No-external training windows:", len(train_dataset_noext))

No-external training windows: 42809


No-external model

In [34]:
model_noext = AutoformerForecastV2(
    context=CONTEXT,
    horizon=HORIZON,
    external_dim=X_norm.shape[1],
    d_model=64,
    n_heads=4,
    n_layers=2,
    kernel_size=25,
    dropout=0.1,
    max_delay=128
).to(device)

num_params_noext = sum(
    p.numel()
    for p in model_noext.parameters()
    if p.requires_grad
)

print("No-external parameters:", num_params_noext)

No-external parameters: 5805821


No-external validation function

In [35]:
def validate_model_noext(model, target, origin):
    model.eval()

    context_start = origin - CONTEXT

    target_context = target[
        context_start:origin
    ]

    target_context_tensor = torch.from_numpy(
        target_context
    ).float().unsqueeze(0).to(device)

    external_context_tensor = torch.zeros(
        1,
        CONTEXT,
        X_norm.shape[1],
        device=device
    )

    external_future_tensor = torch.zeros(
        1,
        HORIZON,
        X_norm.shape[1],
        device=device
    )

    with torch.no_grad():
        prediction = model(
            target_context_tensor,
            external_context_tensor,
            external_future_tensor
        )

    return prediction.squeeze(0).cpu().numpy()

Train no-external model

In [36]:
set_seed(42)

optimizer = torch.optim.AdamW(
    model_noext.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

history_noext, best_epoch_noext, best_rmse_noext = train_model_v2(
    model_noext,
    train_loader_noext,
    EPOCHS
)

print()
print("Best no-external epoch:", best_epoch_noext)
print("Best no-external RMSE:", best_rmse_noext)

Epoch 01/20 | Loss 0.98035 | MAE 72.288 | sMAPE 97.993 | RMSE 83.548
  -> New best model
Epoch 02/20 | Loss 0.93082 | MAE 74.162 | sMAPE 98.665 | RMSE 85.331
Epoch 03/20 | Loss 0.86096 | MAE 87.022 | sMAPE 100.980 | RMSE 101.476
Epoch 04/20 | Loss 0.77810 | MAE 98.093 | sMAPE 101.751 | RMSE 118.826
Epoch 05/20 | Loss 0.70383 | MAE 96.781 | sMAPE 102.759 | RMSE 116.003
Early stopping at epoch 5. Best epoch: 1

Best no-external epoch: 1
Best no-external RMSE: 83.54819


Save ablation results

In [37]:
ablation_results = pd.DataFrame([
    {
        "model": "Autoformer + external",
        "seed": 42,
        "best_epoch": best_epoch_v2,
        "MAE": float(
            history_v2.loc[
                history_v2["RMSE"].idxmin(), "MAE"
            ]
        ),
        "sMAPE": float(
            history_v2.loc[
                history_v2["RMSE"].idxmin(), "sMAPE"
            ]
        ),
        "RMSE": float(best_rmse_v2),
        "parameters": num_params
    },
    {
        "model": "Autoformer - external",
        "seed": 42,
        "best_epoch": best_epoch_noext,
        "MAE": float(
            history_noext.loc[
                history_noext["RMSE"].idxmin(), "MAE"
            ]
        ),
        "sMAPE": float(
            history_noext.loc[
                history_noext["RMSE"].idxmin(), "sMAPE"
            ]
        ),
        "RMSE": float(best_rmse_noext),
        "parameters": num_params_noext
    }
])

display(ablation_results)

,model,seed,best_epoch,MAE,sMAPE,RMSE,parameters
0,Autoformer + external,42,2,39.351246,71.941711,52.478088,5805821
1,Autoformer - external,42,1,72.288338,97.993484,83.548187,5805821


without external, seeds 123 and 2026

In [47]:
NOEXT_SEEDS = [123, 2026]

noext_seed_results = []

for seed in NOEXT_SEEDS:
    print(f"\n{'='*60}")
    print(f"Training no-external model with seed {seed}")
    print(f"{'='*60}")

    set_seed(seed)

    # Fresh no-external Autoformer
    seed_model_noext = AutoformerForecastV2(
        context=CONTEXT,
        horizon=HORIZON,
        external_dim=X_norm.shape[1],
        d_model=64,
        n_heads=4,
        n_layers=2,
        kernel_size=25,
        dropout=0.1,
        max_delay=128
    ).to(device)

    # Fresh optimizer and scheduler
    seed_optimizer = torch.optim.AdamW(
        seed_model_noext.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    seed_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        seed_optimizer,
        T_max=EPOCHS
    )

    # train_model_v2 uses the global optimizer/scheduler,
    # so temporarily replace them.
    optimizer_backup = optimizer
    scheduler_backup = scheduler

    optimizer = seed_optimizer
    scheduler = seed_scheduler

    seed_history_noext, seed_best_epoch_noext, seed_best_rmse_noext = train_model_v2(
        seed_model_noext,
        train_loader_noext,
        EPOCHS
    )

    # Restore original optimizer/scheduler
    optimizer = optimizer_backup
    scheduler = scheduler_backup

    # Get metrics from the best epoch
    best_row = seed_history_noext.loc[
        seed_history_noext["RMSE"].idxmin()
    ]

    noext_seed_results.append({
        "model": "Autoformer - external",
        "seed": seed,
        "best_epoch": int(seed_best_epoch_noext),
        "MAE": float(best_row["MAE"]),
        "sMAPE": float(best_row["sMAPE"]),
        "RMSE": float(best_row["RMSE"]),
        "parameters": sum(
            p.numel()
            for p in seed_model_noext.parameters()
            if p.requires_grad
        )
    })

    print(f"\nSeed {seed} complete.")
    print(f"Best epoch: {seed_best_epoch_noext}")
    print(f"Best RMSE: {seed_best_rmse_noext:.5f}")


Training no-external model with seed 123
Epoch 01/20 | Loss 0.97275 | MAE 72.919 | sMAPE 99.755 | RMSE 82.372
  -> New best model
Epoch 02/20 | Loss 0.90136 | MAE 77.513 | sMAPE 97.940 | RMSE 91.882
Epoch 03/20 | Loss 0.81752 | MAE 102.218 | sMAPE 102.841 | RMSE 125.954
Epoch 04/20 | Loss 0.73740 | MAE 88.806 | sMAPE 100.741 | RMSE 106.236
Epoch 05/20 | Loss 0.66519 | MAE 100.865 | sMAPE 106.223 | RMSE 116.980
Early stopping at epoch 5. Best epoch: 1

Seed 123 complete.
Best epoch: 1
Best RMSE: 82.37202

Training no-external model with seed 2026
Epoch 01/20 | Loss 0.97556 | MAE 71.445 | sMAPE 98.700 | RMSE 81.236
  -> New best model
Epoch 02/20 | Loss 0.89304 | MAE 81.776 | sMAPE 101.348 | RMSE 94.532
Epoch 03/20 | Loss 0.80444 | MAE 87.496 | sMAPE 104.508 | RMSE 102.411
Epoch 04/20 | Loss 0.72105 | MAE 78.629 | sMAPE 100.882 | RMSE 91.916
Epoch 05/20 | Loss 0.65050 | MAE 67.321 | sMAPE 94.759 | RMSE 81.067
  -> New best model
Epoch 06/20 | Loss 0.59621 | MAE 69.081 | sMAPE 95.551 | R

In [49]:
ablation_external = pd.DataFrame(all_seed_results)

ablation_all_seed_results = pd.concat(
    [
        ablation_external,
        pd.DataFrame(noext_seed_results)
    ],
    ignore_index=True
)

display(ablation_all_seed_results)

,model,seed,best_epoch,MAE,sMAPE,RMSE,parameters
0,Autoformer + external,42,2,39.351246,71.941711,52.478088,5805821
1,Autoformer + external,123,16,30.332438,66.088898,36.750431,5805821
2,Autoformer + external,2026,7,29.837692,62.915184,39.130836,5805821
3,Autoformer - external,123,1,72.919006,99.754646,82.372025,5805821
4,Autoformer - external,2026,5,67.320885,94.758827,81.067375,5805821


In [52]:
noext_seed42 = pd.DataFrame([
    {
        "model": "Autoformer - external",
        "seed": 42,
        "best_epoch": 1,
        "MAE": 72.288338,
        "sMAPE": 97.993484,
        "RMSE": 83.548187,
        "parameters": 5805821
    }
])

ablation_all_seed_results = pd.concat(
    [
        ablation_external,
        noext_seed42,
        pd.DataFrame(noext_seed_results)
    ],
    ignore_index=True
)

display(ablation_all_seed_results)

,model,seed,best_epoch,MAE,sMAPE,RMSE,parameters
0,Autoformer + external,42,2,39.351246,71.941711,52.478088,5805821
1,Autoformer + external,123,16,30.332438,66.088898,36.750431,5805821
2,Autoformer + external,2026,7,29.837692,62.915184,39.130836,5805821
3,Autoformer - external,42,1,72.288338,97.993484,83.548187,5805821
4,Autoformer - external,123,1,72.919006,99.754646,82.372025,5805821
5,Autoformer - external,2026,5,67.320885,94.758827,81.067375,5805821


In [53]:
ABLATION_PATH = "/content/drive/MyDrive/colab/AI651_Task2/external_ablation.csv"

ablation_all_seed_results.to_csv(
    ABLATION_PATH,
    index=False
)

print("Saved:", ABLATION_PATH)
print()
display(ablation_all_seed_results)

Saved: /content/drive/MyDrive/colab/AI651_Task2/external_ablation.csv



,model,seed,best_epoch,MAE,sMAPE,RMSE,parameters
0,Autoformer + external,42,2,39.351246,71.941711,52.478088,5805821
1,Autoformer + external,123,16,30.332438,66.088898,36.750431,5805821
2,Autoformer + external,2026,7,29.837692,62.915184,39.130836,5805821
3,Autoformer - external,42,1,72.288338,97.993484,83.548187,5805821
4,Autoformer - external,123,1,72.919006,99.754646,82.372025,5805821
5,Autoformer - external,2026,5,67.320885,94.758827,81.067375,5805821


complete ablation table

In [51]:
ablation_all_seed_results = pd.concat(
    [
        ablation_results,
        pd.DataFrame(noext_seed_results)
    ],
    ignore_index=True
)

display(ablation_all_seed_results)

,model,seed,best_epoch,MAE,sMAPE,RMSE,parameters
0,Autoformer + external,42,2,39.351246,71.941711,52.478088,5805821
1,Autoformer - external,42,1,72.288338,97.993484,83.548187,5805821
2,Autoformer - external,123,1,72.919006,99.754646,82.372025,5805821
3,Autoformer - external,2026,5,67.320885,94.758827,81.067375,5805821


Save

In [38]:
ablation_results.to_csv(
    "/content/drive/MyDrive/colab/AI651_Task2/external_ablation.csv",
    index=False
)

print("Ablation results saved.")

Ablation results saved.


Multi-seed experiment

In [39]:
SEEDS = [123, 2026]

seed_results = []

for seed in SEEDS:

    print("\n" + "=" * 70)
    print(f"TRAINING SEED {seed}")
    print("=" * 70)

    set_seed(seed)

    seed_model = AutoformerForecastV2(
        context=CONTEXT,
        horizon=HORIZON,
        external_dim=X_norm.shape[1],
        d_model=64,
        n_heads=4,
        n_layers=2,
        kernel_size=25,
        dropout=0.1,
        max_delay=128
    ).to(device)

    seed_optimizer = torch.optim.AdamW(
        seed_model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    seed_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        seed_optimizer,
        T_max=EPOCHS
    )

    # Temporarily use these inside the training function
    optimizer_backup = optimizer
    scheduler_backup = scheduler

    optimizer = seed_optimizer
    scheduler = seed_scheduler

    seed_history, seed_best_epoch, seed_best_rmse = (
        train_model_v2(
            seed_model,
            train_loader,
            EPOCHS
        )
    )

    optimizer = optimizer_backup
    scheduler = scheduler_backup

    best_row = seed_history.loc[
        seed_history["RMSE"].idxmin()
    ]

    seed_results.append({
        "model": "Autoformer + external",
        "seed": seed,
        "best_epoch": int(seed_best_epoch),
        "MAE": float(best_row["MAE"]),
        "sMAPE": float(best_row["sMAPE"]),
        "RMSE": float(best_row["RMSE"]),
        "parameters": num_params
    })

    print(
        f"\nSeed {seed} best: "
        f"MAE={best_row['MAE']:.3f}, "
        f"sMAPE={best_row['sMAPE']:.3f}, "
        f"RMSE={best_row['RMSE']:.3f}"
    )


TRAINING SEED 123
Epoch 01/20 | Loss 0.60618 | MAE 45.452 | sMAPE 81.494 | RMSE 55.465
  -> New best model
Epoch 02/20 | Loss 0.38822 | MAE 46.526 | sMAPE 82.960 | RMSE 55.921
Epoch 03/20 | Loss 0.30957 | MAE 40.989 | sMAPE 75.413 | RMSE 49.721
  -> New best model
Epoch 04/20 | Loss 0.26579 | MAE 49.594 | sMAPE 84.843 | RMSE 56.455
Epoch 05/20 | Loss 0.23705 | MAE 46.631 | sMAPE 84.776 | RMSE 53.458
Epoch 06/20 | Loss 0.21620 | MAE 36.992 | sMAPE 72.662 | RMSE 44.098
  -> New best model
Epoch 07/20 | Loss 0.20095 | MAE 45.870 | sMAPE 80.245 | RMSE 51.988
Epoch 08/20 | Loss 0.18906 | MAE 41.989 | sMAPE 78.915 | RMSE 48.706
Epoch 09/20 | Loss 0.17946 | MAE 38.753 | sMAPE 75.608 | RMSE 45.198
Epoch 10/20 | Loss 0.17170 | MAE 37.389 | sMAPE 72.382 | RMSE 43.111
  -> New best model
Epoch 11/20 | Loss 0.16528 | MAE 34.954 | sMAPE 70.307 | RMSE 41.882
  -> New best model
Epoch 12/20 | Loss 0.16006 | MAE 32.265 | sMAPE 67.564 | RMSE 38.451
  -> New best model
Epoch 13/20 | Loss 0.15526 | MAE 

In [40]:
all_seed_results = pd.concat(
    [
        ablation_results[
            ablation_results["model"] == "Autoformer + external"
        ],
        pd.DataFrame(seed_results)
    ],
    ignore_index=True
)

display(all_seed_results)

,model,seed,best_epoch,MAE,sMAPE,RMSE,parameters
0,Autoformer + external,42,2,39.351246,71.941711,52.478088,5805821
1,Autoformer + external,123,16,30.332438,66.088898,36.750431,5805821
2,Autoformer + external,2026,7,29.837692,62.915184,39.130836,5805821


In [41]:
all_seed_results.to_csv(
    "/content/drive/MyDrive/colab/AI651_Task2/multi_seed_results.csv",
    index=False
)

print("Multi-seed results saved.")

Multi-seed results saved.


Multiple validation origins

In [42]:
VALIDATION_ORIGINS = [
    VAL_START - 3 * HORIZON,
    VAL_START - 2 * HORIZON,
    VAL_START - HORIZON,
    VAL_START
]

rolling_results = []

for origin in VALIDATION_ORIGINS:

    prediction_norm = validate_model(
        model,
        y_norm,
        X_norm,
        origin
    )

    actual = y[
        origin:origin + HORIZON
    ]

    prediction = (
        prediction_norm * y_std
        + y_mean
    )

    metrics = calculate_metrics(
        actual,
        prediction
    )

    rolling_results.append({
        "origin": origin,
        "start_time_idx": origin + 1,
        "end_time_idx": origin + HORIZON,
        **metrics
    })

rolling_results = pd.DataFrame(
    rolling_results
)

display(rolling_results)

,origin,start_time_idx,end_time_idx,MAE,sMAPE,RMSE
0,42984,42985,43152,44.607616,84.446838,58.229393
1,43152,43153,43320,50.442291,81.318077,63.400597
2,43320,43321,43488,39.618172,81.409554,51.939934
3,43488,43489,43656,39.351246,71.941711,52.478088


In [43]:
print("Best epoch:", best_epoch_v2)
print("Best RMSE:", best_rmse_v2)

best_row = history_v2.loc[history_v2["RMSE"].idxmin()]
display(best_row)

Best epoch: 2
Best RMSE: 52.47809


,1
epoch,2.000000
train_loss,0.394283
MAE,39.351246
sMAPE,71.941711
RMSE,52.478088


In [44]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for f in files:
        if "v2" in f.lower() or "ablation" in f.lower() or "multi_seed" in f.lower():
            print(os.path.join(root, f))

/content/drive/MyDrive/colab/AI651_Task2/multi_seed_results.csv
/content/drive/MyDrive/colab/AI651_Task2/external_ablation.csv
/content/drive/MyDrive/colab/AI651_Task2/v2_history.csv
/content/drive/MyDrive/colab/AI651_Task2/v2_best.pt


In [45]:
ORIGINAL_CKPT = "/content/drive/MyDrive/colab/AI651_Task2/v2_best.pt"

ckpt = torch.load(
    ORIGINAL_CKPT,
    map_location=device,
    weights_only=False
)

model.load_state_dict(ckpt["model_state_dict"])
model.eval()

print("Original checkpoint loaded.")
print("Best epoch:", ckpt["best_epoch"])
print("Best RMSE:", ckpt["best_rmse"])
print("Parameters:", ckpt["num_params"])

Original checkpoint loaded.
Best epoch: 2
Best RMSE: 52.47809
Parameters: 5805821


In [46]:
prediction_norm = validate_model(
    model,
    y_norm,
    X_norm,
    VAL_START
)

prediction = prediction_norm * y_std + y_mean

metrics = calculate_metrics(
    val_y,
    prediction
)

print(metrics)

{'MAE': np.float32(39.351246), 'sMAPE': np.float32(71.94171), 'RMSE': np.float32(52.47809)}


Create the full-history training loader

In [54]:
# Final training uses all observed target values.
# The hidden 168-point horizon is NOT used as target data.

TRAIN_END_FULL = VAL_START

train_dataset_full = ForecastDataset(
    target=y_norm,
    external_features=X_norm,
    context=CONTEXT,
    horizon=HORIZON,
    start=0,
    end=TRAIN_END_FULL
)

train_loader_full = DataLoader(
    train_dataset_full,
    batch_size=BATCH_SIZE,
    shuffle=True
)

print("Full-history training dataset created.")
print("Observed target points:", TRAIN_END_FULL)
print("Forecast horizon:", HORIZON)
print("External feature dimensions:", X_norm.shape[1])
print("Training windows:", len(train_dataset_full))

Full-history training dataset created.
Observed target points: 43488
Forecast horizon: 168
External feature dimensions: 10
Training windows: 42809


In [55]:
# Correct full-history training boundary.
# All 43,656 observed target values are available for final training.

TRAIN_END_FULL = 43656

train_dataset_full = ForecastDataset(
    target=y_norm,
    external_features=X_norm,
    context=CONTEXT,
    horizon=HORIZON,
    start=0,
    end=TRAIN_END_FULL
)

train_loader_full = DataLoader(
    train_dataset_full,
    batch_size=BATCH_SIZE,
    shuffle=True
)

print("Full-history training dataset created.")
print("Observed target points:", TRAIN_END_FULL)
print("Hidden forecast horizon:", HORIZON)
print("External feature dimensions:", X_norm.shape[1])
print("Training windows:", len(train_dataset_full))

Full-history training dataset created.
Observed target points: 43656
Hidden forecast horizon: 168
External feature dimensions: 10
Training windows: 42977


Create the final Autoformer

In [56]:
# Final model: Autoformer + external features
# Use seed 123 because it achieved the best validation RMSE.

FINAL_SEED = 123
FINAL_EPOCHS = 16

set_seed(FINAL_SEED)

final_model = AutoformerForecastV2(
    context=CONTEXT,
    horizon=HORIZON,
    external_dim=X_norm.shape[1],
    d_model=64,
    n_heads=4,
    n_layers=2,
    kernel_size=25,
    dropout=0.1,
    max_delay=128
).to(device)

final_optimizer = torch.optim.AdamW(
    final_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

final_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    final_optimizer,
    T_max=FINAL_EPOCHS
)

print("Final model created.")
print("Seed:", FINAL_SEED)
print("Epochs:", FINAL_EPOCHS)
print(
    "Trainable parameters:",
    sum(p.numel() for p in final_model.parameters() if p.requires_grad)
)

Final model created.
Seed: 123
Epochs: 16
Trainable parameters: 5805821


Train the final model

In [57]:
# Train final model on all observed data

optimizer_backup = optimizer
scheduler_backup = scheduler

optimizer = final_optimizer
scheduler = final_scheduler

final_history, final_best_epoch, final_best_rmse = train_model_v2(
    final_model,
    train_loader_full,
    FINAL_EPOCHS
)

# Restore the previous optimizer/scheduler
optimizer = optimizer_backup
scheduler = scheduler_backup

print()
print("Final training complete.")
print("Best epoch:", final_best_epoch)
print("Best RMSE:", final_best_rmse)

Epoch 01/16 | Loss 0.60462 | MAE 43.589 | sMAPE 76.574 | RMSE 55.162
  -> New best model
Epoch 02/16 | Loss 0.38591 | MAE 38.018 | sMAPE 73.900 | RMSE 46.858
  -> New best model
Epoch 03/16 | Loss 0.30879 | MAE 31.314 | sMAPE 64.889 | RMSE 38.857
  -> New best model
Epoch 04/16 | Loss 0.26527 | MAE 29.742 | sMAPE 61.203 | RMSE 38.409
  -> New best model
Epoch 05/16 | Loss 0.23714 | MAE 29.511 | sMAPE 62.378 | RMSE 36.953
  -> New best model
Epoch 06/16 | Loss 0.21689 | MAE 30.770 | sMAPE 64.826 | RMSE 37.911
Epoch 07/16 | Loss 0.20189 | MAE 24.121 | sMAPE 57.402 | RMSE 31.238
  -> New best model
Epoch 08/16 | Loss 0.18984 | MAE 20.793 | sMAPE 56.593 | RMSE 27.585
  -> New best model
Epoch 09/16 | Loss 0.18020 | MAE 20.896 | sMAPE 58.406 | RMSE 27.465
  -> New best model
Epoch 10/16 | Loss 0.17351 | MAE 24.414 | sMAPE 57.703 | RMSE 31.457
Epoch 11/16 | Loss 0.16757 | MAE 21.784 | sMAPE 58.568 | RMSE 28.385
Epoch 12/16 | Loss 0.16292 | MAE 19.980 | sMAPE 56.628 | RMSE 26.415
  -> New bes

Generate the 168 hidden forecasts


In [58]:
# Generate the final 168-step forecast

final_model.eval()

with torch.no_grad():
    # Last observed context
    context_target = torch.tensor(
        y_norm[-CONTEXT:],
        dtype=torch.float32,
        device=device
    ).unsqueeze(0)

    # External features for the context + forecast horizon
    context_external = torch.tensor(
        X_norm[-CONTEXT:],
        dtype=torch.float32,
        device=device
    ).unsqueeze(0)

    future_external = torch.tensor(
        X_norm[43656:43656 + HORIZON],
        dtype=torch.float32,
        device=device
    ).unsqueeze(0)

    final_prediction_norm = final_model(
        context_target,
        context_external,
        future_external
    )

    final_prediction = (
        final_prediction_norm.squeeze().cpu().numpy()
        * y_std
        + y_mean
    )

print("Forecast generated.")
print("Number of predictions:", len(final_prediction))
print("First 5:", final_prediction[:5])
print("Last 5:", final_prediction[-5:])

Forecast generated.
Number of predictions: 168
First 5: [30.937363  5.580841 15.416901 11.664619 12.670776]
Last 5: [80.39294  79.36247  78.00821  71.4945   63.706635]


Verify the final forecast

In [59]:
# Final forecast sanity checks

forecast_df = pd.DataFrame({
    "time_idx": np.arange(43657, 43825),
    "forecast": final_prediction
})

print("Number of rows:", len(forecast_df))
print("First time_idx:", forecast_df["time_idx"].iloc[0])
print("Last time_idx:", forecast_df["time_idx"].iloc[-1])
print("NaN values:", forecast_df["forecast"].isna().sum())
print("Inf values:", np.isinf(forecast_df["forecast"]).sum())
print()
display(forecast_df.head())
display(forecast_df.tail())

Number of rows: 168
First time_idx: 43657
Last time_idx: 43824
NaN values: 0
Inf values: 0



,time_idx,forecast
0,43657,30.937363
1,43658,5.580841
2,43659,15.416901
3,43660,11.664619
4,43661,12.670776


,time_idx,forecast
163,43820,80.392937
164,43821,79.362473
165,43822,78.008209
166,43823,71.494499
167,43824,63.706635


In [60]:
# Create the exact comma-separated leaderboard submission

submission_values = ",".join(
    f"{x:.6f}" for x in final_prediction
)

print("Number of forecast values:", len(submission_values.split(",")))
print()
print("Leaderboard submission:")
print(submission_values)

Number of forecast values: 168

Leaderboard submission:
30.937363,5.580841,15.416901,11.664619,12.670776,7.938576,19.086525,22.545601,21.705666,21.519623,22.748138,36.633976,36.404507,34.922855,40.596954,50.624737,58.404366,57.944672,70.844574,93.604950,94.312469,99.206551,102.641747,89.995415,95.075386,89.671158,79.263741,82.293938,82.916718,75.751221,60.870415,57.256500,56.998997,63.949554,65.425507,78.509392,88.803513,95.927864,117.483025,124.323105,140.752930,154.245758,182.607315,180.787109,186.531723,192.791626,180.918243,181.572479,176.697327,160.785461,156.994446,137.205032,127.203712,109.404076,104.847862,85.345940,81.063095,69.563126,60.916016,58.301163,62.490582,75.600311,99.203369,98.239296,106.822067,124.999809,134.379379,156.985443,159.978638,166.708862,167.971588,164.735840,178.768097,165.538422,158.799225,155.455551,134.079544,142.405334,123.849426,112.102135,96.382500,93.877357,76.961578,57.885422,63.419014,61.242100,44.406166,41.978344,54.807968,44.367016,65.639381,79

In [61]:
# Save the exact leaderboard submission string to Google Drive

submission_path = "/content/drive/MyDrive/colab/AI651_Task2/leaderboard_submission.txt"

with open(submission_path, "w") as f:
    f.write(submission_values)

print("Saved:", submission_path)
print("Values:", len(submission_values.split(",")))

Saved: /content/drive/MyDrive/colab/AI651_Task2/leaderboard_submission.txt
Values: 168


In [62]:
# Final verification of the saved leaderboard submission

with open(submission_path, "r") as f:
    saved_submission = f.read().strip()

saved_values = saved_submission.split(",")

print("File exists:", os.path.exists(submission_path))
print("Number of values:", len(saved_values))
print("All values numeric:", all(np.isfinite(float(x)) for x in saved_values))
print("Contains spaces:", " " in saved_submission)
print("First value:", saved_values[0])
print("Last value:", saved_values[-1])
print("Time indices covered: 43657–43824")

File exists: True
Number of values: 168
All values numeric: True
Contains spaces: False
First value: 30.937363
Last value: 63.706635
Time indices covered: 43657–43824


In [63]:
# Save the final trained Autoformer checkpoint

final_model_path = "/content/drive/MyDrive/colab/AI651_Task2/final_model.pt"

torch.save(
    {
        "model_state_dict": final_model.state_dict(),
        "seed": FINAL_SEED,
        "epochs": FINAL_EPOCHS,
        "best_epoch": final_best_epoch,
        "best_training_window_rmse": final_best_rmse,
        "trainable_parameters": sum(
            p.numel() for p in final_model.parameters()
            if p.requires_grad
        ),
        "context": CONTEXT,
        "horizon": HORIZON,
        "external_dim": X_norm.shape[1],
        "d_model": 64,
        "n_heads": 4,
        "n_layers": 2,
        "kernel_size": 25,
        "dropout": 0.1,
        "max_delay": 128,
        "y_mean": y_mean,
        "y_std": y_std,
    },
    final_model_path
)

print("Saved:", final_model_path)
print("Trainable parameters:", sum(
    p.numel() for p in final_model.parameters()
    if p.requires_grad
))
print("Final training epochs:", FINAL_EPOCHS)
print("Best training-window epoch:", final_best_epoch)

Saved: /content/drive/MyDrive/colab/AI651_Task2/final_model.pt
Trainable parameters: 5805821
Final training epochs: 16
Best training-window epoch: 16


 Save final training history to Google Drive

In [64]:
# Save final training history to Google Drive

final_history_path = "/content/drive/MyDrive/colab/AI651_Task2/final_history.csv"

pd.DataFrame(final_history).to_csv(final_history_path, index=False)

print("Saved:", final_history_path)
print("Training epochs recorded:", len(final_history))

Saved: /content/drive/MyDrive/colab/AI651_Task2/final_history.csv
Training epochs recorded: 16
